# DPE on Waterbirds with corrected labels: fresh backbones, both label sets

Select **GPU T4 x2**, turn **Internet on**, then **Save Version -> Save & Run All**. Optional but
recommended: **Add Input -> Notebook Output -> `noerula/dpe-reproduction`** so the two official
ERM backbones of seeds 5 and 6 are reused (their SHA-256 is checked) instead of retrained.

Three CUB species (021 Eastern Towhee, 088 Western Meadowlark, 102 Western Wood-Pewee) are labelled
waterbird in Waterbirds only because the generator matches `tern` inside `eastern`/`western`.
This run crosses **backbone training labels** (official / corrected, 72 training images differ)
with **DPE validation labels** (official / corrected, 18 images) on fresh SubpopBench ERM backbones,
and scores every model on **official and corrected test labels** (90 images).

Pipeline: the audited runner of `noerula/dpe-reproduction` (SubpopBench ERM, default
hyper-parameters, 5,001 steps; the authors' unmodified DPE `main.py`, 15 heads, `wga_val`), with
a label-set switch. Upstream code is pinned and unmodified.

**Declared before the run** (also in `summarize.py`): primary contrast = (official backbone,
corrected validation labels) minus (official backbone, official validation labels) on corrected
test labels, averaged over head seeds within a backbone and then over backbone seeds 5-8.
Secondary: corrected backbone + corrected validation labels, and corrected backbone only; every
contrast is also reported on official test labels. No test result chooses any setting.

The notebook first runs a ~20 minute smoke test and stops if it fails. Work is scheduled on both
GPUs in priority order and stops launching new jobs after `TIME_LIMIT_HOURS`, so the version
saves before Kaggle's 12-hour limit. `report/REPORT.md` is refreshed after every finished job.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, shutil, glob, hashlib
T0 = time.time()
WORK = Path('/kaggle/working/dpe-labels')
WORK.mkdir(parents=True, exist_ok=True)
RUNNER = WORK / 'runner.py'
SUMMARIZE = WORK / 'summarize.py'
RESULTS = WORK / 'results'
REPORT_DIR = WORK / 'report'
SMOKE_DIR = WORK / 'smoke'
RESULTS.mkdir(parents=True, exist_ok=True)

REUSE_SEEDS = [5, 6]          # official backbones reused from noerula/dpe-reproduction when attached
NEW_SEEDS = [7, 8]            # fresh backbones for both label sets (declared before the run)
HEAD_OFFSETS = [0, 1000, 2000]  # DPE head seeds = backbone seed + offset
TRAIN_ATTR = 'yes'            # group-annotated validation (DPE Table 2 setting)
WORKERS_PER_JOB = 2
TIME_LIMIT_HOURS = 11.0       # no new job starts if it would not finish by then
RUN_SMOKE_TEST = True
EXPECTED_REUSE_SHA = {
    5: 'ead5ae8118eba69a31a96cf54c6e34bd78fa78a382a97f1824fb11f5822bdaef',
    6: 'e088b07b34911d3a569e252ed807490e4380cc2474cb8d36a6184537c40fb2f1',
}
REGIMES = [('official', 'official'), ('official', 'corrected'), ('corrected', 'official'), ('corrected', 'corrected')]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def hours():
    return (time.time() - T0) / 3600

print(subprocess.check_output(['nvidia-smi'], text=True))

## Pinned training environment (same as `noerula/dpe-reproduction`)

In [ ]:
BOOT = WORK / 'bootstrap'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--target', str(BOOT), 'uv==0.9.8'], check=True)
UV_ENV = dict(os.environ, PYTHONPATH=str(BOOT), UV_CACHE_DIR=str(WORK / 'uv-cache'),
              UV_PYTHON_INSTALL_DIR=str(WORK / 'python'))
VENV = WORK / 'venv'
PY = VENV / 'bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', str(VENV)], env=UV_ENV, check=True)
ISOLATED_ENV = dict(os.environ, PYTHONNOUSERSITE='1')
for key in ('PYTHONHOME', 'PYTHONPATH'):
    ISOLATED_ENV.pop(key, None)
subprocess.run([str(PY), '-m', 'pip', 'install', '--quiet', 'torch==2.6.0+cu118',
                'torchvision==0.21.0+cu118', '--index-url', 'https://download.pytorch.org/whl/cu118'], env=ISOLATED_ENV, check=True)
PINS = ['Pillow==10.3.0', 'jsonargparse==4.37.0', 'matplotlib==3.8.3', 'netcal==1.3.5',
        'numpy==1.26.4', 'pandas==2.2.3', 'scikit-learn==1.3.2', 'scipy==1.12.0',
        'seaborn==0.13.2', 'tqdm==4.66.2', 'transformers==4.46.3', 'wandb==0.19.1',
        'timm==1.0.12', 'tensorboard==2.18.0', 'tensorboard-logger==0.1.0', 'protobuf==3.20.3']
subprocess.run([str(PY), '-m', 'pip', 'install', '--quiet', *PINS], env=ISOLATED_ENV, check=True)
subprocess.run([str(PY), '-m', 'pip', 'check'], env=ISOLATED_ENV, check=True)
BASE_ENV = dict(ISOLATED_ENV, TORCH_HOME=str(WORK / 'cache/torch'), HF_HOME=str(WORK / 'cache/hf'),
                MPLCONFIGDIR=str(WORK / 'cache/matplotlib'), MPLBACKEND='Agg',
                OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1')
subprocess.run([str(PY), '-c', "import torch, torchvision; print(torch.__version__, torchvision.__version__); "
                "assert torch.cuda.device_count() == 2, 'Choose Kaggle T4 x2'; "
                "print([torch.cuda.get_device_name(i) for i in range(2)])"], env=BASE_ENV, check=True)

## Pinned upstream code, official data, and a corrected-label copy

The corrected copy shares the same image files (symlink) and changes only `y` for the 180 images
of the three species (72 train, 18 validation, 90 test). Each worker re-verifies group counts.

In [ ]:
import urllib.request, tarfile
def checkout(url, destination, commit):
    if not destination.exists():
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    current = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if current != commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    assert subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip() == commit
    assert not subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain', '--untracked-files=no'], text=True)

DPE = WORK / 'dpe-upstream'
SUBPOP = WORK / 'subpopbench-upstream'
checkout('https://github.com/minhto2802/dpe4subpop.git', DPE, '68b811461e238d44cced6ae6c8ce75b66cad5449')
checkout('https://github.com/YyzHarry/SubpopBench.git', SUBPOP, '4d3dbbe21029666ef19d040e110ec22908640c5b')
DATA_OFF = WORK / 'data_official'
WB = DATA_OFF / 'waterbirds'
WB.mkdir(parents=True, exist_ok=True)
IMAGE_ROOT = WB / 'waterbird_complete95_forest2water2'
if not (IMAGE_ROOT / 'metadata.csv').exists():
    archive = WORK / 'waterbirds.tar.gz'
    partial = WORK / 'waterbirds.tar.gz.part'
    if not archive.exists():
        errors = []
        for base in ('https://downloads.cs.stanford.edu/nlp/data/dro/', 'https://nlp.stanford.edu/data/dro/'):
            try:
                urllib.request.urlretrieve(base + 'waterbird_complete95_forest2water2.tar.gz', partial)
                partial.replace(archive)
                break
            except Exception as e:
                errors.append(str(e))
        else:
            raise RuntimeError('Waterbirds download failed: ' + str(errors))
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            candidate = (WB / member.name).resolve()
            if not candidate.is_relative_to(WB.resolve()) or member.issym() or member.islnk() or member.isdev():
                raise RuntimeError('Unexpected archive entry: ' + member.name)
        tar.extractall(WB)
format_code = "\nimport pandas as pd, sys\nfrom pathlib import Path\nwb = Path(sys.argv[1])\ndf = pd.read_csv(wb / 'waterbird_complete95_forest2water2/metadata.csv')\ndf['filename'] = df['img_filename']\ndf['a'] = df['place']\ndf.to_csv(wb / 'metadata_waterbirds.csv', index=False)\nprint(df.groupby(['split','y','a']).size())\n"
subprocess.run([str(PY), '-c', format_code, str(WB)], env=BASE_ENV, check=True)

DATA_COR = WORK / 'data_corrected'
WBC = DATA_COR / 'waterbirds'
WBC.mkdir(parents=True, exist_ok=True)
link = WBC / 'waterbird_complete95_forest2water2'
if not link.exists():
    link.symlink_to(IMAGE_ROOT.resolve(), target_is_directory=True)
correct_code = (
    "import pandas as pd, sys\n"
    "src, dst = sys.argv[1], sys.argv[2]\n"
    "df = pd.read_csv(src)\n"
    "species = ('021.Eastern_Towhee/', '088.Western_Meadowlark/', '102.Western_Wood_Pewee/')\n"
    "mask = df['filename'].str.startswith(species)\n"
    "assert int(mask.sum()) == 180 and (df.loc[mask, 'y'] == 1).all(), int(mask.sum())\n"
    "df.loc[mask, 'y'] = 0\n"
    "df.to_csv(dst, index=False)\n"
    "print('relabelled', int(mask.sum()), 'images'); print(df.groupby(['split','y','a']).size())\n")
subprocess.run([str(PY), '-c', correct_code, str(WB / 'metadata_waterbirds.csv'), str(WBC / 'metadata_waterbirds.csv')],
               env=BASE_ENV, check=True)
DATA = {'official': DATA_OFF, 'corrected': DATA_COR}
subprocess.run([str(PY), '-m', 'subpopbench.train', '--help'], cwd=SUBPOP, env=BASE_ENV, check=True, stdout=subprocess.DEVNULL)
subprocess.run([str(PY), 'main.py', '--help'], cwd=DPE, env=BASE_ENV, check=True, stdout=subprocess.DEVNULL)
subprocess.run([str(PY), '-c', "import torchvision; torchvision.models.resnet50(weights='IMAGENET1K_V1')"], env=BASE_ENV, check=True)

## Runner (audited runner of `noerula/dpe-reproduction` plus a label-set switch) and scorer

In [ ]:
%%writefile "{RUNNER}"
#!/usr/bin/env python3
"""Audited DPE orchestration with a label-set switch (official | corrected Waterbirds labels).

Derived from the runner of the public notebook noerula/dpe-reproduction. Changes, all marked
"LABEL-SET" or "SMOKE": expected group counts per label set; --steps / --num-stages for a smoke
test only; head-count checks follow --num-stages. Upstream training code is unchanged.

Training functions are loaded from pinned upstream sources, not reimplemented here.
See docs/dpe_reproduction_audit.md for protocol limitations.
"""
import argparse
import ast
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import importlib.util
from types import SimpleNamespace

DPE_COMMIT = "68b811461e238d44cced6ae6c8ce75b66cad5449"
SUBPOP_COMMIT = "4d3dbbe21029666ef19d040e110ec22908640c5b"
# LABEL-SET: (y, a) counts per split, order (0,0), (0,1), (1,0), (1,1). "corrected" relabels the
# three species that generate_waterbirds.py marks waterbird through the substring "tern"
# (021 Eastern Towhee, 088 Western Meadowlark, 102 Western Wood-Pewee) as landbirds.
GROUP_COUNTS_BY_LABEL_SET = {
    "official": {0: [3498, 184, 56, 1057], 1: [467, 466, 133, 133], 2: [2255, 2255, 642, 642]},
    "corrected": {0: [3502, 252, 52, 989], 1: [478, 473, 122, 126], 2: [2298, 2302, 599, 595]},
}
GROUP_COUNTS = GROUP_COUNTS_BY_LABEL_SET["official"]


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    tmp.replace(path)


def verify_repo(path, commit):
    actual = subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    if actual != commit:
        raise RuntimeError(f"Expected {commit}, found {actual}: {path}")
    if subprocess.check_output(["git", "-C", str(path), "status", "--porcelain", "--untracked-files=no"], text=True):
        raise RuntimeError(f"Upstream tracked files have been modified: {path}")


def verify_data(data_dir, label_set="official"):
    import pandas as pd
    GROUP_COUNTS = GROUP_COUNTS_BY_LABEL_SET[label_set]  # LABEL-SET
    root = Path(data_dir) / "waterbirds"
    meta = root / "metadata_waterbirds.csv"
    df = pd.read_csv(meta)
    if (len(df) != sum(map(sum, GROUP_COUNTS.values())) or
            df[["filename", "y", "a", "split"]].isna().any().any() or
            set(df.y) != {0, 1} or set(df.a) != {0, 1}):
        raise ValueError("Expected the complete original Waterbirds dataset with binary labels/attributes")
    if set(df.split) != {0, 1, 2} or df.filename.duplicated().any():
        raise ValueError("Unexpected splits or duplicate image paths")
    for s, expected in GROUP_COUNTS.items():
        observed = df[df.split == s].groupby(["y", "a"]).size()
        counts = [int(observed.get((y, a), 0)) for y in (0, 1) for a in (0, 1)]
        if counts != expected:
            raise ValueError(f"Split {s}: expected {expected}, got {counts}")
    image_root = root / "waterbird_complete95_forest2water2"
    image_hash = hashlib.sha256()
    for name in df.filename:
        p = image_root / name
        if not p.is_file():
            raise FileNotFoundError(p)
        image_hash.update(name.encode() + b"\0" + bytes.fromhex(sha256(p)))
    return {"metadata_sha256": sha256(meta), "images_sha256": image_hash.hexdigest(),
            "group_counts": GROUP_COUNTS, "rows": len(df), "label_set": label_set}


def load_functions(dpe_root, names, namespace):
    """Compile only named definitions; never execute notebook cells or magics."""
    path = Path(dpe_root) / "notebooks/01_waterbirds_with_attribute_annotation.ipynb"
    nb = json.loads(path.read_text())
    found = {}
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        source = "\n".join(line for line in source.splitlines() if not line.startswith(("%", "!")))
        for node in ast.parse(source).body:
            if isinstance(node, ast.FunctionDef) and node.name in names:
                found[node.name] = node
    if set(found) != set(names):
        raise ValueError(f"Missing upstream definitions: {set(names) - set(found)}")
    module = ast.Module(body=list(found.values()), type_ignores=[])
    exec(compile(module, str(path), "exec"), namespace)
    return namespace


def load_official_main(dpe_root):
    spec = importlib.util.spec_from_file_location("dpe_official_main", Path(dpe_root) / "main.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def stage1_command(args, out):
    """The Waterbirds train_pe.sh + train_all_pe.sh invocation, without Slurm/W&B."""
    return [sys.executable, "main.py", "--dataset_name", "Waterbirds", "--model_name", "resnet50",
            "--data_dir", str(args.data_dir), "--pretrained_path", str(out / "ckpt_last.pt"),
            "--ckpt_dir", str(out / "stage1"), "--seed", str(args.seed), "--device", "cuda",
            "--loss_name", "isomax", "--stage", "1", "--num_stages", str(args.num_stages), "--epochs", "20",
            "--train_mode", "freeze", "--train_attr", args.train_attr, "--subsample_type", "group",
            "--batch_size", "256", "--batch_size_eval", "512", "--workers", str(args.workers),
            "--optim", "sgd", "--lr", "0.001", "--momentum", "0.9", "--optim_weight_decay", "0",
            "--scheduler", "none", "--wd_weight", "10", "--cov_reg", "500000", "--entropic_scale", "30",
            "--norm_emb", "yes", "--ensemble_criterion", "wga_val", "--wdb_group", "dpe",
            "-ncbt", "-sit", "--no_wandb"]


def official_environment(dpe_root):
    """Avoid jsonargparse's implicit environment arguments overriding defaults."""
    env = dict(os.environ)
    for node in ast.walk(ast.parse((Path(dpe_root) / "main.py").read_text())):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "add_argument":
            for argument in node.args:
                if isinstance(argument, ast.Constant) and isinstance(argument.value, str) and argument.value.startswith("--"):
                    env.pop(argument.value[2:].upper(), None)
    return env


def run_official_cli(args, out, datasets, device):
    """Train with the original CLI, then evaluate its saved final ensemble."""
    import torch
    from torch.utils.data import DataLoader
    from utils.misc import fix_random_seed
    from utils.models import get_model

    stage_dir = out / "stage1"
    stage_dir.mkdir()
    cmd = stage1_command(args, out)
    write_json(out / "stage1_command.json", {"argv": cmd, "cwd": str(args.dpe_root)})
    with open(out / "stage1.log", "w") as log:
        subprocess.run(cmd, cwd=args.dpe_root, env=official_environment(args.dpe_root),
                       stdout=log, stderr=subprocess.STDOUT, check=True)
    prototypes = torch.load(stage_dir / "prototype_ensemble_wga_val.pt", map_location="cpu", weights_only=True)
    scales = torch.load(stage_dir / "dist_scales_wga_val.pt", map_location="cpu", weights_only=True)
    heads = args.num_stages - 1  # SMOKE: 15 heads unless --num-stages is lowered
    if prototypes.shape != (2, heads, 2048) or len(scales) != heads:
        raise ValueError(f"Official CLI did not produce a {heads}-head, 2048-dimensional ensemble")
    if not torch.isfinite(prototypes).all() or not all(s.shape == (1,) and torch.isfinite(s).all() for s in scales):
        raise ValueError("Nonfinite prototypes or invalid distance scales in the official output")
    ensemble = [[prototypes[:, i:i+1].to(device), s.to(device)] for i, s in enumerate(scales)]
    torch.save([[p.cpu(), s.cpu()] for p, s in ensemble], out / "ensemble.pt")

    # Replay the original NumPy subsampler solely to save exact membership/order.
    # Only dataset construction consumes NumPy randomness in the original CLI;
    # torch initializers/DataLoaders affect the separate torch RNG stream.
    fix_random_seed(args.seed, True, False)
    from utils import datasets as dsets
    subsets = []
    for stage in range(1, args.num_stages):
        ds = dsets.Waterbirds(str(args.data_dir), "va", None, train_attr=args.train_attr,
                             subsample_type="group", stage=stage, pre_extracted_feats=datasets["val"].feats)
        subsets.append({"stage": stage, "files": ds.x, "labels": ds.y, "groups": ds.g,
                        "cov_reg": 5e5, "batches_per_epoch": len(ds) // 256})
    write_json(out / "subsets.json", subsets)
    model = get_model("Waterbirds", 2, "freeze", str(out / "ckpt_last.pt"),
                      loss_name="isomax", pretrained_imgnet=False).to(device)
    test_loader = DataLoader(datasets["test"], batch_size=512, shuffle=False, num_workers=args.workers)
    # Use the released evaluator directly; no custom ensemble arithmetic.
    logits = load_official_main(args.dpe_root).evaluate_ensemble_fixed_backbone(
        prototypes.to(device), test_loader, model, distance_scales=[s.to(device) for s in scales], return_preds=True)
    probabilities = logits.softmax(2).mean(0).numpy()
    return probabilities


def metric_summary(probabilities, dataset):
    import numpy as np
    labels, groups = np.asarray(dataset.y), np.asarray(dataset.g)
    probabilities = np.asarray(probabilities)
    if (probabilities.shape != (len(labels), 2) or not np.isfinite(probabilities).all() or
            (probabilities < 0).any() or (probabilities > 1).any() or
            not np.allclose(probabilities.sum(1), 1, atol=1e-6) or
            set(groups) != {"00", "01", "10", "11"}):
        raise ValueError("Expected finite normalized binary probabilities and all four Waterbirds groups")
    prediction = probabilities.argmax(1)
    correct = prediction == labels
    per_group = {g: {"n": int((groups == g).sum()),
                     "correct": int(correct[groups == g].sum()),
                     "accuracy": float(correct[groups == g].mean())} for g in sorted(set(groups))}
    return {"overall_accuracy": float(correct.mean()),
            "wga": min(v["accuracy"] for v in per_group.values()), "per_group": per_group}


def convert_checkpoint(source, target):
    """Strictly map known DPE or SubpopBench ResNet50/linear checkpoints."""
    import torch
    import torchvision
    from torch import nn
    # Only load checkpoints the user trusts; SubpopBench includes argparse objects.
    ckpt = torch.load(source, map_location="cpu", weights_only=False)
    if "model_dict" in ckpt:
        state = ckpt["model_dict"]
        prefix = "featurizer.network."
        children = {"conv1": "0", "bn1": "1", "layer1": "4", "layer2": "5", "layer3": "6", "layer4": "7"}
        mapped = {}
        for k, v in state.items():
            if k.startswith(prefix):
                name, rest = k[len(prefix):].split(".", 1)
                mapped[f"0.{children[name]}.{rest}"] = v
        for k in ("weight", "bias"):
            mapped["2." + k] = state["classifier." + k]
    else:
        mapped = ckpt
    net = torchvision.models.resnet50(weights=None)
    model = nn.Sequential(nn.Sequential(*list(net.children())[:-1]), nn.Flatten(), nn.Linear(2048, 2))
    model.load_state_dict(mapped, strict=True)
    torch.save(model.state_dict(), target)
    return model


def run_stage0(args, out):
    target = out / "ckpt_last.pt"
    if args.protocol == "checkpoint":
        if not args.checkpoint or not args.checkpoint_provenance:
            raise ValueError("checkpoint mode requires a trusted --checkpoint and --checkpoint-provenance")
        model = convert_checkpoint(args.checkpoint, target)
        return model, {"source": str(args.checkpoint), "sha256": sha256(args.checkpoint),
                       "provenance": args.checkpoint_provenance, "selection": "supplied checkpoint"}
    if args.protocol == "subpopbench":
        # Original entry point, default hyperparameters, fixed 5001 steps; no ES.
        # This is a new default-hparam run, NOT the paper authors' recovered checkpoint.
        bootstrap = ("import runpy; from subpopbench.dataset.datasets import Waterbirds; "
                     f"Waterbirds.N_WORKERS={args.workers}; "
                     "runpy.run_module('subpopbench.train', run_name='__main__')")
        cmd = [sys.executable, "-c", bootstrap, "--algorithm", "ERM", "--dataset", "Waterbirds",
               "--train_attr", "no", "--data_dir", str(args.data_dir), "--output_dir", str(out / "subpopbench"),
               "--output_folder_name", "standard", "--seed", str(args.seed), "--hparams_seed", "0"]
        if args.steps:  # SMOKE only; the real run keeps SubpopBench's default 5,001 steps
            cmd += ["--steps", str(args.steps)]
        cwd = args.subpop_root
    else:
        epochs = 100 if args.protocol == "legacy100" else 400
        cmd = [sys.executable, "main.py", "--dataset_name", "Waterbirds", "--data_dir", str(args.data_dir),
               "--ckpt_dir", str(out / "native"), "--pretrained_imgnet", "--loss_name", "ce",
               "--epochs", str(epochs), "--lr", "0.001", "--batch_size", "128", "--batch_size_eval", "256" if args.protocol == "legacy100" else "512",
               "--workers", str(args.workers), "--no_wandb", "--seed", str(args.seed)]
        if args.protocol == "legacy100":
            cmd.append("--no_augmentation")
        cwd = args.dpe_root
        (out / "native").mkdir()
    write_json(out / "stage0_command.json", {"argv": cmd, "cwd": str(cwd)})
    with open(out / "stage0.log", "w") as f:
        subprocess.run(cmd, cwd=cwd, env=official_environment(args.dpe_root), stdout=f,
                       stderr=subprocess.STDOUT, check=True)
    if args.protocol == "subpopbench":
        source = out / "subpopbench/standard_attrNo" / f"Waterbirds_ERM_hparams0_seed{args.seed}/model.pkl"
        if not (source.parent / "done").exists():
            raise RuntimeError("SubpopBench did not finish")
    else:
        source = out / "native/ckpt_last.pt"
    model = convert_checkpoint(source, target)
    return model, {"source": str(source), "sha256": sha256(source), "selection": "fixed final checkpoint",
                   "note": "Fresh run; paper's selected ERM artifact/hparams not recovered"}


def extract_features(model, dataset, device, workers):
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    model.eval()
    features = []
    with torch.no_grad():
        for _, inputs, *_ in DataLoader(dataset, batch_size=256, shuffle=False, num_workers=workers):
            features.append(model[:-1](inputs.to(device)).cpu().numpy())
    raw = np.concatenate(features)
    if raw.shape != (len(dataset), 2048) or not np.isfinite(raw).all() or (raw.std(1) == 0).any():
        raise ValueError("Invalid raw features")
    return raw


def fit_dpe(args, out, model, device):
    import numpy as np
    import torch
    import torchvision
    from torch import nn
    from torch.utils.data import DataLoader
    from tqdm import tqdm
    from utils import datasets as dsets
    from utils.misc import fix_random_seed
    from utils.eval_helpers import eval_metrics
    from utils.isomaxplus import IsoMaxPlusLossFirstPart, IsoMaxPlusLossSecondPart

    datasets, raw_features = {}, {}
    fix_random_seed(args.seed, True, False)  # Deterministic convolution for raw feature extraction.
    model = model.to(device)
    for name, split in (("val", "va"), ("test", "te")):
        datasets[name] = dsets.Waterbirds(str(args.data_dir), split, None)
        if args.head_implementation == "official_cli":
            extraction_args = SimpleNamespace(dataset_name="Waterbirds", train_mode="freeze",
                                              pretrained_path=str(out / "ckpt_last.pt"), device=device,
                                              model_name="resnet50", workers=args.workers, batch_size_eval=512)
            raw = load_official_main(args.dpe_root).extract_features(extraction_args, datasets[name])
        else:
            raw = extract_features(model, datasets[name], device, args.workers)
        if raw.shape != (len(datasets[name]), 2048) or not np.isfinite(raw).all() or (raw.std(1) == 0).any():
            raise ValueError("Invalid raw features")
        raw_features[name] = raw
        np.save(out / f"feats_{name}.npy", raw)
        # NumPy ddof=0, per image. This is the upstream DPE normalization.
        datasets[name].feats = (raw - raw.mean(1, keepdims=True)) / raw.std(1, keepdims=True)

    with torch.no_grad():
        raw_logits = model[-1](torch.from_numpy(raw_features["test"]).to(device))
        erm = metric_summary(raw_logits.softmax(1).cpu().numpy(), datasets["test"])
    write_json(out / "erm_metrics_raw_features.json", erm)

    if args.head_implementation == "official_cli":
        probs = run_official_cli(args, out, datasets, device)
        return save_predictions(out, probs, datasets["test"])

    namespace = dict(torch=torch, torchvision=torchvision, nn=nn, np=np, DataLoader=DataLoader, tqdm=tqdm,
                     dsets=dsets, eval_metrics=eval_metrics, IsoMaxPlusLossFirstPart=IsoMaxPlusLossFirstPart,
                     IsoMaxPlusLossSecondPart=IsoMaxPlusLossSecondPart, data_dir=str(args.data_dir),
                     workers=args.workers, batch_size_train=256)
    load_functions(args.dpe_root, ["evaluate", "get_subsampled_train_set", "get_train_loader",
                                  "init_model", "train_prototypes", "cov_reg_scheduler_inv"], namespace)
    # Upstream evaluate/train_prototypes default to cuda. Isolation maps the process
    # to one GPU. CPU is supported only in unit smoke tests, with this default bound.
    upstream_evaluate = namespace["evaluate"]
    namespace["evaluate"] = lambda m, loader: upstream_evaluate(m, loader, device=device)
    val_loader = DataLoader(datasets["val"], batch_size=256, shuffle=False, num_workers=args.workers)

    fix_random_seed(args.seed)  # Same reset and cudnn flags as the upstream notebook.
    train_loader = namespace["get_train_loader"](datasets, args.train_attr)
    scheduler = namespace["cov_reg_scheduler_inv"](5e5, 15)
    ensemble, full_model = [], None
    subsets = []
    for stage in range(1, 16):
        ds = train_loader.dataset
        # Preserve membership AND order, since minibatch/drop_last behavior matters.
        subsets.append({"stage": stage, "files": ds.x, "labels": ds.y, "groups": ds.g,
                        "cov_reg": scheduler(stage), "batches_per_epoch": len(train_loader)})
        write_json(out / "subsets.json", subsets)
        full_model = namespace["init_model"](str(out / "ckpt_last.pt"), _model=full_model, device=device)
        chosen = namespace["train_prototypes"](
            train_loader, val_loader, full_model, _prototype_ensemble=ensemble,
            epochs=20, cov_reg=scheduler(stage), wd_weight=10, entropic=30, lr=1e-3,
            _stage=stage, verbose=False, device=device)
        if chosen is None or not all(torch.isfinite(t).all() for t in chosen):
            raise RuntimeError(f"No finite selected head at stage {stage}")
        ensemble.append(chosen)
        torch.save([[p.cpu(), s.cpu()] for p, s in ensemble], out / "ensemble.pt")
        # The upstream demo evaluates on test after every stage; replace this with
        # an equally shaped VAL inference to preserve DataLoader RNG consumption.
        # Its model-head assignment is preserved before initializing the next head.
        ensemble_probabilities(ensemble, val_loader, full_model, device, args.dpe_root)
        train_loader = namespace["get_train_loader"](datasets, args.train_attr)

    # Fixed 15 heads, then a single final test inference. No best-test-head count.
    test_loader = DataLoader(datasets["test"], batch_size=256, shuffle=False, num_workers=args.workers)
    probs = ensemble_probabilities(ensemble, test_loader, full_model, device, args.dpe_root)
    return save_predictions(out, probs, datasets["test"])


def save_predictions(out, probs, dataset):
    import numpy as np
    result = metric_summary(probs, dataset)
    np.savez_compressed(out / "test_predictions.npz", probabilities=probs,
                        y=np.asarray(dataset.y), groups=np.asarray(dataset.g), filenames=np.asarray(dataset.x))
    return result


def ensemble_probabilities(ensemble, loader, model, device, dpe_root):
    """Run the original notebook evaluator; capture its probability array unchanged."""
    import torch
    import numpy as np
    from torch import nn
    from tqdm import tqdm
    from utils.eval_helpers import eval_metrics, get_acc
    captured = {}
    def capture(probabilities, *labels):
        captured["probabilities"] = probabilities
        return eval_metrics(probabilities, *labels)
    namespace = dict(torch=torch, nn=nn, np=np, tqdm=tqdm, eval_metrics=capture, get_acc=get_acc)
    load_functions(dpe_root, ["evaluate_ensemble"], namespace)
    namespace["evaluate_ensemble"](ensemble, loader, model, device=device, verbose=False)
    return captured["probabilities"]


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--dpe-root", type=Path, required=True)
    p.add_argument("--subpop-root", type=Path, required=True)
    p.add_argument("--data-dir", type=Path, required=True)
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--protocol", choices=["subpopbench", "checkpoint", "legacy100", "ermstar400"], default="subpopbench")
    p.add_argument("--head-implementation", choices=["official_cli", "official_notebook"], default="official_cli")
    p.add_argument("--train-attr", choices=["yes", "no"], default="yes")
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--checkpoint", type=Path)
    p.add_argument("--checkpoint-provenance", default="")
    p.add_argument("--label-set", choices=["official", "corrected"], required=True)  # LABEL-SET
    p.add_argument("--steps", type=int, default=0)  # SMOKE only
    p.add_argument("--num-stages", type=int, default=16)  # SMOKE only
    args = p.parse_args()
    for name in ("dpe_root", "subpop_root", "data_dir", "output_dir"):
        setattr(args, name, getattr(args, name).resolve())
    verify_repo(args.dpe_root, DPE_COMMIT)
    verify_repo(args.subpop_root, SUBPOP_COMMIT)
    args.output_dir.mkdir(parents=True, exist_ok=False)  # Never silently reuse stale seed artifacts.
    import torch
    import torchvision
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Launch with CUDA_VISIBLE_DEVICES=<one GPU>; expected exactly one visible CUDA device")
    manifest = {"args": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},
                "source": {"dpe": DPE_COMMIT, "subpopbench": SUBPOP_COMMIT, "runner_sha256": sha256(__file__)},
                "environment": {"python": sys.version, "torch": torch.__version__, "torchvision": torchvision.__version__,
                                "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
                                "gpu": torch.cuda.get_device_name(0), "visible_devices": os.getenv("CUDA_VISIBLE_DEVICES")},
                "dataset": verify_data(args.data_dir, args.label_set), "status": "running"}
    write_json(args.output_dir / "manifest.json", manifest)
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    (args.output_dir / "pip-freeze.txt").write_text(freeze)
    # Native SubpopBench checkpoints pickle their EarlyStopping class alongside
    # model_dict, even without early stopping. Make that trusted module available.
    sys.path.insert(0, str(args.subpop_root))
    try:
        model, provenance = run_stage0(args, args.output_dir)
        manifest["backbone"] = provenance
        manifest["checkpoint_sha256"] = sha256(args.output_dir / "ckpt_last.pt")
        write_json(args.output_dir / "manifest.json", manifest)
        sys.path.insert(0, str(args.dpe_root))
        result = fit_dpe(args, args.output_dir, model, "cuda")
        write_json(args.output_dir / "metrics.json", result)
        manifest["artifacts"] = {name: sha256(args.output_dir / name) for name in
                                 ("ensemble.pt", "feats_val.npy", "feats_test.npy", "subsets.json", "test_predictions.npz")}
        if args.head_implementation == "official_cli":
            for name in ("stage1/prototype_ensemble_wga_val.pt", "stage1/dist_scales_wga_val.pt", "stage1_command.json"):
                manifest["artifacts"][name] = sha256(args.output_dir / name)
        manifest["status"] = "complete"
        print(json.dumps(result), flush=True)
    except BaseException as e:
        manifest["status"] = "failed"
        manifest["error"] = repr(e)
        raise
    finally:
        write_json(args.output_dir / "manifest.json", manifest)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile "{SUMMARIZE}"
"""Score every finished job on official and corrected test labels and write the declared contrasts.

Each job directory holds test_predictions.npz (probabilities, y, groups, filenames) written by the
runner, plus job.json describing backbone seed, backbone label set, validation label set and head
seed. The test labels in the npz follow the job's validation label set; both label sets are
rebuilt here from the three species names, so every job is scored on both.

Declared before the run (do not edit after results are read):
  primary   : (backbone official, val corrected) - (backbone official, val official), corrected test,
              mean over head seeds within a backbone, then over backbones.
  secondary : (backbone corrected, val corrected) - (backbone official, val official), corrected test;
              (backbone corrected, val official) - (backbone official, val official), corrected test;
              all contrasts also on official test labels.
"""
import csv
import json
import statistics
import sys
from pathlib import Path

import numpy as np

SPECIES = ("021.Eastern_Towhee", "088.Western_Meadowlark", "102.Western_Wood_Pewee")
GROUPS = ("00", "01", "10", "11")
REGIMES = (("official", "official"), ("official", "corrected"), ("corrected", "official"), ("corrected", "corrected"))
T95 = {1: 12.706204736, 2: 4.302652730, 3: 3.182446305, 4: 2.776445105, 5: 2.570581836}
CONTRASTS = [(("official", "corrected"), ("official", "official"), "primary: fix validation labels"),
             (("corrected", "corrected"), ("official", "official"), "fix training and validation labels"),
             (("corrected", "official"), ("official", "official"), "fix training labels only"),
             (("corrected", "corrected"), ("official", "corrected"), "add training fix to validation fix")]


def affected(filenames):
    return np.array([any(s in str(f) for s in SPECIES) for f in filenames])


def score(pred, y, a):
    g = np.array([f"{yy}{aa}" for yy, aa in zip(y, a)])
    acc = {k: float((pred[g == k] == y[g == k]).mean()) * 100 for k in GROUPS}
    return dict(wga=min(acc.values()), overall=float((pred == y).mean()) * 100, groups=acc)


def score_job(job_dir):
    job = json.loads((job_dir/"job.json").read_text())
    with np.load(job_dir/"test_predictions.npz", allow_pickle=False) as z:
        p, y, groups, files = z["probabilities"], z["y"].astype(int), z["groups"].astype(str), z["filenames"]
    if len(y) != 5794 or p.shape != (5794, 2):
        raise ValueError(f"{job_dir}: expected 5,794 test predictions")
    mask = affected(files)
    if int(mask.sum()) != 90:
        raise ValueError(f"{job_dir}: expected 90 affected test images, found {int(mask.sum())}")
    a = np.array([int(g[1]) for g in groups])
    pred = p.argmax(1)
    y_off, y_cor = y.copy(), y.copy()
    y_off[mask], y_cor[mask] = 1, 0
    return dict(job, official=score(pred, y_off, a), corrected=score(pred, y_cor, a),
                affected_predicted_landbird=int((pred[mask] == 0).sum()))


def summarize(results, out):
    results, out = Path(results), Path(out)
    rows, failures = [], []
    for d in sorted(p for p in results.iterdir() if (p/"job.json").exists()):
        if not (d/"test_predictions.npz").exists():
            failures.append(d.name)
            continue
        try:
            rows.append(score_job(d))
        except Exception as e:  # keep going; report the failure
            failures.append(f"{d.name}: {e!r}")
    out.mkdir(parents=True, exist_ok=True)
    flat = [dict(job=r["name"], backbone_seed=r["backbone_seed"], backbone_labels=r["backbone_labels"],
                 val_labels=r["val_labels"], head_seed=r["head_seed"], test_labels=t, wga=r[t]["wga"],
                 overall=r[t]["overall"], **{f"g{k}": v for k, v in r[t]["groups"].items()},
                 affected_predicted_landbird=r["affected_predicted_landbird"])
            for r in rows for t in ("official", "corrected")]
    if flat:
        with (out/"summary.csv").open("w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(flat[0])); w.writeheader(); w.writerows(flat)

    def cell(seed, regime, test):
        v = [r[test]["wga"] for r in rows if r["backbone_seed"] == seed and (r["backbone_labels"], r["val_labels"]) == regime]
        return statistics.mean(v) if v else None, len(v)

    seeds = sorted({r["backbone_seed"] for r in rows})
    lines = ["# Label-corrected backbones: results", "",
             f"Jobs scored: {len(rows)}. Missing or failed: {len(failures)}.", ""]
    if failures:
        lines += ["Failures:", *[f"- {x}" for x in failures], ""]
    for test in ("corrected", "official"):
        lines += [f"## Test labels: {test}", "", "WGA (%) per backbone seed, mean over head seeds (n head seeds in brackets).", "",
                  "| Backbone labels / val labels | " + " | ".join(f"seed {s}" for s in seeds) + " |",
                  "| --- |" + " ---: |" * len(seeds)]
        for regime in REGIMES:
            vals = [cell(s, regime, test) for s in seeds]
            lines.append(f"| {regime[0]} / {regime[1]} | " + " | ".join("" if m is None else f"{m:.2f} ({n})" for m, n in vals) + " |")
        lines += ["", "| Contrast | Mean (pp) | 95% t interval | Backbones | Per backbone |", "| --- | ---: | --- | ---: | --- |"]
        for a_, b_, label in CONTRASTS:
            d = []
            for s in seeds:
                x, y = cell(s, a_, test)[0], cell(s, b_, test)[0]
                if x is not None and y is not None:
                    d.append((s, x-y))
            if not d:
                lines.append(f"| {label} | | | 0 | |"); continue
            vals = [v for _, v in d]
            m = statistics.mean(vals)
            ci = ""
            if len(vals) >= 2:
                h = T95[len(vals)-1]*statistics.stdev(vals)/len(vals)**.5
                ci = f"[{m-h:+.2f}, {m+h:+.2f}]"
            lines.append(f"| {label} | {m:+.2f} | {ci} | {len(vals)} | " + ", ".join(f"s{s}: {v:+.2f}" for s, v in d) + " |")
        lines.append("")
    (out/"REPORT.md").write_text("\n".join(lines))
    return "\n".join(lines)


if __name__ == "__main__":
    print(summarize(sys.argv[1], sys.argv[2]))


## Reuse the official seed-5 and seed-6 backbones when their outputs are attached

In [ ]:
BACKBONES = {}   # (label_set, seed) -> dict(ckpt=Path, provenance=str)
REUSED = set()
for s in REUSE_SEEDS:
    found = None
    for p in sorted(glob.glob(f'/kaggle/input/**/seed{s}/ckpt_last.pt', recursive=True)):
        if sha256(p) == EXPECTED_REUSE_SHA[s]:
            found = Path(p); break
    if found:
        BACKBONES[('official', s)] = dict(ckpt=found, provenance=f'official SubpopBench ERM backbone seed {s} reused from noerula/dpe-reproduction ({found}); sha256 verified')
        REUSED.add(s)
    print(f'seed {s}:', 'reusing ' + str(found) if found else 'not attached; it will be retrained')

## Job scheduler and smoke test

`train` jobs train a backbone on one label set and fit DPE with the same labels (head seed = backbone
seed). `heads` jobs fit DPE on an existing backbone with either label set and any head seed.

In [ ]:
import signal
sys.path.insert(0, str(WORK))
import importlib; summarize = importlib.import_module('summarize')

def train_job(labels, seed, extra=()):
    return dict(kind='train', name=f'train-{labels}-s{seed}', backbone_labels=labels, val_labels=labels,
                backbone_seed=seed, head_seed=seed, extra=list(extra))

def heads_job(bb, seed, val, head, extra=()):
    return dict(kind='heads', name=f'heads-bb{bb}-s{seed}-val{val}-h{head}', backbone_labels=bb, val_labels=val,
                backbone_seed=seed, head_seed=head, extra=list(extra))

def launch(job, gpu, results):
    out = results / job['name']
    cmd = [str(PY), str(RUNNER), '--dpe-root', str(DPE), '--subpop-root', str(SUBPOP),
           '--data-dir', str(DATA[job['val_labels']]), '--output-dir', str(out), '--seed', str(job['head_seed']),
           '--train-attr', TRAIN_ATTR, '--head-implementation', 'official_cli', '--workers', str(WORKERS_PER_JOB),
           '--label-set', job['val_labels']]
    if job['kind'] == 'train':
        cmd += ['--protocol', 'subpopbench']
    else:
        bb = BACKBONES[(job['backbone_labels'], job['backbone_seed'])]
        cmd += ['--protocol', 'checkpoint', '--checkpoint', str(bb['ckpt']), '--checkpoint-provenance', bb['provenance']]
    cmd += job['extra']
    env = dict(BASE_ENV, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONHASHSEED=str(job['head_seed']))
    log = open(results / f"{job['name']}.log", 'w')
    proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    return dict(job=job, proc=proc, log=log, gpu=gpu, start=time.time(), out=out)

def finish(run, results):
    run['log'].close()
    job, out = run['job'], run['out']
    seconds = time.time() - run['start']
    record = {k: v for k, v in job.items() if k != 'extra'}
    record.update(returncode=run['proc'].returncode, seconds=seconds, gpu=run['gpu'])
    out.mkdir(parents=True, exist_ok=True)
    (out / 'job.json').write_text(json.dumps(record, indent=1))
    ok = run['proc'].returncode == 0 and (out / 'test_predictions.npz').exists()
    if ok and job['kind'] == 'train':
        BACKBONES[(job['backbone_labels'], job['backbone_seed'])] = dict(
            ckpt=out / 'ckpt_last.pt', provenance=f"trained in this notebook ({job['name']})")
    if not ok:
        print(f"FAILED {job['name']} (rc={run['proc'].returncode}); last log lines:")
        print((results / f"{job['name']}.log").read_text(errors='replace')[-3000:])
    return ok, seconds

def stop(run):
    if run['proc'].poll() is None:
        os.killpg(run['proc'].pid, signal.SIGTERM)
        try:
            run['proc'].wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(run['proc'].pid, signal.SIGKILL); run['proc'].wait()

def wait_all(runs, results):
    results_ok = {}
    while any(r['proc'].poll() is None for r in runs):
        time.sleep(30)
    for r in runs:
        results_ok[r['job']['name']] = finish(r, results)[0]
    return results_ok

if RUN_SMOKE_TEST:
    if SMOKE_DIR.exists():
        shutil.rmtree(SMOKE_DIR)
    SMOKE_DIR.mkdir(parents=True)
    smoke_train = train_job('corrected', 99, extra=['--steps', '30', '--num-stages', '3'])
    smoke_train['name'] = 'smoke-train'
    ok = wait_all([launch(smoke_train, 0, SMOKE_DIR)], SMOKE_DIR)
    assert ok['smoke-train'], 'Smoke training job failed: see the log above. Nothing else was started.'
    saved = dict(BACKBONES)
    smoke_heads = heads_job('corrected', 99, 'official', 1099, extra=['--num-stages', '3'])
    smoke_heads['name'] = 'smoke-heads'
    ok = wait_all([launch(smoke_heads, 1, SMOKE_DIR)], SMOKE_DIR)
    assert ok['smoke-heads'], 'Smoke heads job failed: see the log above.'
    BACKBONES.clear(); BACKBONES.update({k: v for k, v in saved.items() if k[1] != 99})
    print(summarize.summarize(SMOKE_DIR, SMOKE_DIR / 'report'))
    print(f'Smoke test passed after {hours():.2f} h')

## Main run (both GPUs, priority order, time-limited)

In [ ]:
PLAN = []
for s in REUSE_SEEDS:
    if s not in REUSED:
        PLAN.append(train_job('official', s))
    PLAN.append(train_job('corrected', s))
for s in REUSE_SEEDS:
    PLAN += [heads_job('official', s, 'corrected', s), heads_job('corrected', s, 'official', s)]
    if s in REUSED:
        PLAN.append(heads_job('official', s, 'official', s))
for s in NEW_SEEDS:
    PLAN += [train_job('official', s), train_job('corrected', s),
             heads_job('official', s, 'corrected', s), heads_job('corrected', s, 'official', s)]
for off in HEAD_OFFSETS[1:]:
    for s in REUSE_SEEDS + NEW_SEEDS:
        PLAN += [heads_job(bb, s, val, s + off) for bb, val in REGIMES]
print(len(PLAN), 'jobs planned:', [j['name'] for j in PLAN])

ESTIMATE = {'train': 2.5 * 3600, 'heads': 0.5 * 3600}
DURATIONS = {'train': [], 'heads': []}
pending, running, failed, skipped = list(PLAN), {}, set(), set()

def expected(kind):
    return max(DURATIONS[kind]) if DURATIONS[kind] else ESTIMATE[kind]

def pick():
    for job in list(pending):
        key = (job['backbone_labels'], job['backbone_seed'])
        if job['kind'] == 'heads' and key not in BACKBONES:
            source = f"train-{key[0]}-s{key[1]}"
            if source in failed or source in skipped:
                pending.remove(job); skipped.add(job['name'])
            continue
        if hours() * 3600 + 1.15 * expected(job['kind']) > TIME_LIMIT_HOURS * 3600:
            continue
        return job
    return None

last_print = 0
try:
    while True:
        for gpu in (0, 1):
            if gpu in running:
                run = running[gpu]
                if run['proc'].poll() is not None:
                    ok, seconds = finish(run, RESULTS)
                    DURATIONS[run['job']['kind']].append(seconds)
                    (failed if not ok else set()).add(run['job']['name'])
                    del running[gpu]
                    summarize.summarize(RESULTS, REPORT_DIR)
                    print(f"[{hours():.2f} h] finished {run['job']['name']} ok={ok} in {seconds/3600:.2f} h", flush=True)
            if gpu not in running:
                job = pick()
                if job is not None:
                    pending.remove(job)
                    running[gpu] = launch(job, gpu, RESULTS)
                    print(f"[{hours():.2f} h] started {job['name']} on GPU {gpu}", flush=True)
        if not running and (not pending or pick() is None):
            break
        if hours() > TIME_LIMIT_HOURS + 0.5:
            print('Hard time limit reached; stopping running jobs.')
            for run in running.values():
                stop(run); finish(run, RESULTS); failed.add(run['job']['name'])
            running.clear()
            break
        if time.time() - last_print > 600:
            last_print = time.time()
            print(f"[{hours():.2f} h] running: {[r['job']['name'] for r in running.values()]}; "
                  f"pending {len(pending)}, failed {len(failed)}, skipped {len(skipped)}", flush=True)
        time.sleep(30)
finally:
    for run in running.values():
        stop(run)
print('Not started (time limit or missing backbone):', [j['name'] for j in pending] + sorted(skipped))
print('Failed:', sorted(failed))

## Report and compact evidence bundle

`report/REPORT.md` and `report/summary.csv` hold the scored results. The bundle keeps manifests,
metrics, logs, per-image test predictions and one feature cache per backbone (for CPU follow-ups
such as DFR-style retraining); backbone weights stay in the working directory.

In [ ]:
import zipfile
print(summarize.summarize(RESULTS, REPORT_DIR))
(REPORT_DIR / 'plan.json').write_text(json.dumps(dict(
    plan=[j['name'] for j in PLAN], failed=sorted(failed), not_started=[j['name'] for j in pending] + sorted(skipped),
    reused=sorted(REUSED), hours=hours()), indent=1))
bundle = WORK / 'label-corrected-evidence.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in list(REPORT_DIR.rglob('*')) + list(RESULTS.rglob('*')) + list(SMOKE_DIR.rglob('*.log')):
        if not p.is_file() or p.suffix in ('.pt', '.pkl') or 'tfevents' in p.name:
            continue
        if p.name.startswith('feats_'):
            job = p.parent.name
            if not (job.startswith('train-') or (job.startswith('heads-bbofficial') and '-valofficial-' in job
                                                  and job.endswith(f"-h{job.split('-s')[1].split('-')[0]}"))):
                continue
        z.write(p, p.relative_to(WORK))
print('Bundle:', bundle, round(bundle.stat().st_size / 1e6, 1), 'MB')
# Free the large environment so the saved output stays small.
for name in ('venv', 'uv-cache', 'python', 'bootstrap', 'cache', 'data_official', 'data_corrected', 'waterbirds.tar.gz'):
    p = WORK / name
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.exists():
        shutil.rmtree(p, ignore_errors=True)
print(f'Done after {hours():.2f} h')